# 06 - Venue locations
Maps each venue to specific location coordinates
Produces geocoding responses stored in the bucket, and the tables tennis_clean.venue_name_map and tennis_clean.venue_locations
Run after 05_match_dates

## Install the pinned libraries

In [1]:

REQUIREMENTS_URL = "https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

ERROR: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt


## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [2]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

MessageError: Error: credential propagation was unsuccessful

## Helpers

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)
MATCHES = f"{PROJECT_ID}.tennis_clean.matches"

## Create the venue list
Weather is requested by latitude and longitude, but the data only has tournament names. This notebook turns each name into a place and then into coordinates, using Open-Meteo's free geocoding API Names are keyed on the cleaned tournament name.


In [ ]:
venues = run(f"""
SELECT name_key,
       MIN(season) AS first_year, MAX(season) AS last_year,
       COUNT(DISTINCT tourney_id) AS editions,
       ARRAY_TO_STRING(ARRAY_AGG(DISTINCT tourney_level), ',') AS levels
FROM `{MATCHES}`
GROUP BY name_key ORDER BY name_key
""")

# Name in the data -> the place to search for
SEARCH = {
    "australian open": "Melbourne", "melbourne 1": "Melbourne", "melbourne 2": "Melbourne",
    "gippsland trophy": "Melbourne", "yarra valley classic": "Melbourne",
    "grampians trophy": "Melbourne", "phillip island trophy": "Melbourne",
    "roland garros": "Paris", "us open": "New York", "wimbledon": "London",
    "queen's club": "London", "japan open tokyo": "Tokyo",
    "'s hertogenbosch": "'s-Hertogenbosch", "s hertogenbosch": "'s-Hertogenbosch",
    "s-hertogenbosch": "'s-Hertogenbosch", "nurnberg": "Nuremberg",
    "cluj napoca": "Cluj-Napoca", "cluj-napoca 1": "Cluj-Napoca", "cluj-napoca 2": "Cluj-Napoca",
    "adelaide 1": "Adelaide", "adelaide 2": "Adelaide", "charleston 1": "Charleston",
    "guadalajara 2": "Guadalajara", "chicago 2": "Chicago", "hua hin 2": "Hua Hin",
    "calvia": "Calvia", "st. petersburg": "Saint Petersburg", "nur-sultan": "Astana",
}
venues["search_name"] = venues["name_key"].map(SEARCH).fillna(venues["name_key"].str.title())

names = sorted(venues["search_name"].unique())
print(len(venues), "tournament names ->", len(names), "places to geocode")

## Reviewed corrections
The first geocoding results were reviewed by hand. For these places the top result was wrong, so the query or the chosen result is set here:
- `REFETCH`: search for different text, or restrict to a country.
- `PICK_INDEX`: use a result other than the first.

In [ ]:
REFETCH = {                                   # search_name -> (text to search, country code or None)
    "Bol": ("Bol", "HR"),
    "Granby": ("Granby", "CA"),
    "Marrakech": ("Marrakesh", None),
    "Quebec City": ("Quebec", "CA"),
    "Stanford": ("Palo Alto", "US"),
    "Mallorca": ("Calvia", None),
}
PICK_INDEX = {"Indian Wells": 1}              # top result is Indio; the town itself is the second result

queries = {n: REFETCH.get(n, (n, None)) for n in names}

## Fetch and save the raw responses
Each file is written once, so re-running does not call the API again for anything already stored.

The API's  output is in `response`, and `request`.`fetched_at_utc` records how and when it was obtained.

In [ ]:
import json, re, time
import requests
from datetime import datetime, timezone
from google.cloud import storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET_NAME)

GEO_URL = "https://geocoding-api.open-meteo.com/v1/search"
GEO_PREFIX = "raw/open_meteo/geocoding"
used_keys = set()      # the files this run actually uses (recorded in the manifest)

def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

def geocode(name, country=None):
    """Return the stored record for this query, fetching (and storing) it only if it is new."""
    key = slug(name) + (f"__{country}" if country else "")
    used_keys.add(key)
    blob = bucket.blob(f"{GEO_PREFIX}/{key}.json")
    if blob.exists():
        return json.loads(blob.download_as_text())

    params = {"name": name, "count": 5, "language": "en", "format": "json"}
    if country:
        params["countryCode"] = country
    r = requests.get(GEO_URL, params=params, timeout=30)
    r.raise_for_status()

    record = {
        "request": {"url": GEO_URL, "params": params},
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "response": r.json(),          # exactly as returned by the API
    }
    blob.upload_from_string(json.dumps(record, indent=1),
                            content_type="application/json", if_generation_match=0)
    time.sleep(0.2)                    # be polite to the free API
    return record

records = {n: geocode(*queries[n]) for n in names}
print("Responses used:", len(records))

## Choose the result for each place

In [ ]:
rows = []
for name in names:
    text, country = queries[name]
    results = records[name]["response"].get("results", [])
    idx = PICK_INDEX.get(name, 0)
    assert len(results) > idx, f"No usable result for {name}"
    r = results[idx]
    if name in REFETCH:
        decision = f"re-searched as '{text}'" + (f" in {country}" if country else "")
    elif name in PICK_INDEX:
        decision = f"result #{idx + 1} chosen by hand"
    else:
        decision = "top result"
    rows.append({
        "search_name": name, "place": r["name"], "region": r.get("admin1"),
        "country_code": r.get("country_code"), "latitude": r["latitude"],
        "longitude": r["longitude"], "elevation_m": r.get("elevation"),
        "timezone": r.get("timezone"), "geonames_id": r.get("id"),
        "population": r.get("population"), "decision": decision,
    })

locations = pd.DataFrame(rows)
locations[["search_name", "place", "region", "country_code", "population", "decision"]]

## Checks
Check the results are sensible (yes)

In [ ]:
print("places:", len(locations))
print("duplicate search names:", locations["search_name"].duplicated().sum())
print("missing coordinates:", locations["latitude"].isna().sum(),
      "| missing timezone:", locations["timezone"].isna().sum())
assert set(venues["search_name"]) <= set(locations["search_name"]), "Some venues have no location"
print(locations["country_code"].value_counts().head(8))
locations.sort_values("elevation_m", ascending=False).head(5)[["place", "country_code", "elevation_m"]]

## Save the tables
Two tables in BigQuery (the name-to-place map and the places with coordinates), and a versioned copy of each in the bucket. If a file ever changes, it is saved as a new version.

In [ ]:
import hashlib

def save_versioned(path, text, content_type="text/csv"):
    """Write a file once. If it already exists with identical content, fine; if different, stop."""
    blob = bucket.blob(path)
    new_hash = hashlib.sha256(text.encode()).hexdigest()
    if blob.exists():
        if hashlib.sha256(blob.download_as_bytes()).hexdigest() == new_hash:
            print(f"{path}: already stored, identical")
        else:
            raise RuntimeError(f"{path} exists with different content. Save this as a new version instead of overwriting.")
    else:
        blob.upload_from_string(text, content_type=content_type, if_generation_match=0)
        print(f"{path}: saved, sha256 {new_hash[:12]}")

name_map = venues[["name_key", "search_name"]]

for table, df in [("venue_name_map", name_map), ("venue_locations", locations)]:
    bq.load_table_from_dataframe(
        df, f"{PROJECT_ID}.tennis_clean.{table}",
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")).result()
    print(table, "rows:", bq.get_table(f"{PROJECT_ID}.tennis_clean.{table}").num_rows)

VERSION = "v1"
save_versioned(f"reference/venue_name_map/{VERSION}/venue_name_map.csv", name_map.to_csv(index=False))
save_versioned(f"reference/venue_locations/{VERSION}/venue_locations.csv", locations.to_csv(index=False))

## Manifest of the geocoding files this run used

In [ ]:
entries = []
for key in sorted(used_keys):
    data = bucket.blob(f"{GEO_PREFIX}/{key}.json").download_as_bytes()
    entries.append({"file": f"{GEO_PREFIX}/{key}.json", "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest()})

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
manifest_path = f"manifests/open_meteo_geocoding_{stamp}.json"
bucket.blob(manifest_path).upload_from_string(
    json.dumps({"endpoint": GEO_URL, "files": entries}, indent=2), content_type="application/json")
print(len(entries), "files recorded in", manifest_path)